In [2]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [23]:
# The unique identifier for  Bank's app on the Google Play Store
CBE_APP_ID = 'com.combanketh.mobilebanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    CBE_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Commercial Bank of Ethiopia App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

Commercial Bank of Ethiopia App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.284285
Total Ratings: 48,256
Total Reviews: 9,303
Installs     : 5,000,000+


In [24]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Awash Bank...")

result, continuation_token = reviews(
    CBE_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Awash Bank...
Collected 500 raw reviews


In [26]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 22026bb2-c9c4-4040-892b-2a77ebee48b7
  userName: Qurbac Yusuf
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjXZNacCirOZhOuVF_izLaMzsbSV91jun3cOHHjRveNDMdNlYc-c
  content: very nice 100%
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-13 06:12:17
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [27]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'CBE Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,22026bb2-c9c4-4040-892b-2a77ebee48b7,very nice 100%,5,2026-05-13 06:12:17,CBE Bank,Google Play
1,dd4ae5b5-f5a4-42e9-a526-5fe9387dd7a4,good,5,2026-05-12 20:02:00,CBE Bank,Google Play
2,e9233eb9-c338-4c80-8f21-26cd31acd65f,Good to use,5,2026-05-12 13:23:58,CBE Bank,Google Play
3,6b0d612d-f9c1-4cac-bb37-b945c97a2e9a,cbe,1,2026-05-12 07:18:40,CBE Bank,Google Play
4,68ff6a46-3659-47f0-a20f-fba5755a3f67,Cbe,4,2026-05-11 18:21:59,CBE Bank,Google Play


In [28]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [29]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

    # What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Rating distribution:
  5 stars:  339  ███████████████████████████████████████████████████████████████████
  4 stars:   41  ████████
  3 stars:   35  ███████
  2 stars:   11  ██
  1 stars:   74  ██████████████
Sample date values (raw):
0   2026-05-13 06:12:17
1   2026-05-12 20:02:00
2   2026-05-12 13:23:58
3   2026-05-12 07:18:40
4   2026-05-11 18:21:59
5   2026-05-11 12:04:31
6   2026-05-11 10:22:25
7   2026-05-11 01:24:09
8   2026-05-10 18:46:00
9   2026-05-10 17:13:18

Date dtype: datetime64[us]


In [30]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [31]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 127
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [32]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-13 06:12:17
  Target format: YYYY-MM-DD (string or date object)


In [33]:
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Starting with: 500 reviews
Removed 0 rows with missing critical data
Remaining: 500 reviews
Removed 0 duplicate reviews
Remaining: 500 reviews


In [34]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-13 06:12:17
1   2026-05-12 20:02:00
2   2026-05-12 13:23:58
dtype: datetime64[us]

After normalization:
0    2026-05-13
1    2026-05-12
2    2026-05-12
dtype: str

Date range: 2026-03-01 to 2026-05-13


In [35]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [36]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [37]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,very nice 100%,5,2026-05-13,CBE Bank,Google Play
1,Good to use,5,2026-05-12,CBE Bank,Google Play
2,cbe,1,2026-05-12,CBE Bank,Google Play
3,good,5,2026-05-12,CBE Bank,Google Play
4,Cbe,4,2026-05-11,CBE Bank,Google Play
5,best and secured,5,2026-05-11,CBE Bank,Google Play
6,best,5,2026-05-11,CBE Bank,Google Play
7,ok,5,2026-05-11,CBE Bank,Google Play
8,posetive,5,2026-05-10,CBE Bank,Google Play
9,good,5,2026-05-10,CBE Bank,Google Play


In [38]:
# Save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/commercial_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: data/processed/commercial_bank_reviews_clean.csv
